# Spatially Variable Genes with Squidpy: Moran's I

Reproducible code for the blog post at [spatiabio.com](https://www.spatiabio.com).

Dataset: 10x Genomics Visium mouse brain demo.

## Setup

In [ ]:
import squidpy as sq
import scanpy as sc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Load and preprocess

In [ ]:
adata = sc.read_h5ad("../../data/anndata/visium_hne_adata.h5ad")
# Or load directly: adata = sq.datasets.visium_hne_adata()
print(f"Shape: {adata.shape}")

In [ ]:
# Filter to top 500 HVGs
sc.pp.highly_variable_genes(adata, n_top_genes=500, flavor="cell_ranger")
adata = adata[:, adata.var.highly_variable].copy()
print(f"After HVG filter: {adata.shape}")

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## Build spatial graph

In [ ]:
sq.gr.spatial_neighbors(adata)
print("Spatial graph built")

## Run Moran's I

> **Windows note**: The `if __name__ == '__main__':` guard is required on Windows due to multiprocessing. On Linux/Mac it is not needed.

In [ ]:
if __name__ == '__main__':
    sq.gr.spatial_autocorr(adata, mode="moran", n_perms=200, n_jobs=1)

    svg_results = adata.uns["moranI"]
    top20 = svg_results.sort_values("I", ascending=False).head(20)
    print("=== Top 20 Spatially Variable Genes ===")
    print(top20[["I", "pval_norm", "pval_norm_fdr_bh"]].to_string())

    svg_results.to_csv("svg_results.csv")
    print("\nSaved to svg_results.csv")

## Results summary

Top SVGs from this run:

| Rank | Gene | Moran's I | Function |
|------|------|-----------|----------|
| 1 | Itpka | 0.674 | IP3 kinase; hippocampus enriched |
| 2 | Fezf1 | 0.634 | Deep cortical layer TF (L5/L6) |
| 3 | Baiap3 | 0.622 | Synaptic scaffolding |
| 4 | Shox2 | 0.611 | Thalamic/hypothalamic marker |
| 5 | Slc30a3 | 0.600 | Zinc transporter; cortex-specific |